# Сравнение Momentum, RMSprop и Adam на задаче линейной регрессии

В ноутбуке:
1. Реализованы оптимизаторы **Momentum**, **RMSprop** и **Adam** (в дополнение к GD/SGD для контекста).
2. На синтетической задаче сравниваются метрики сходимости и точности.
3. Проводятся эксперименты для разных размерностей признакового пространства и формулируются эмпирические выводы.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
plt.style.use('seaborn-v0_8')

## 1) Вспомогательные функции и синтетические данные

In [ ]:
def make_regression_data(n_samples: int, n_features: int, noise_std: float = 0.1, seed: int = 0):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(n_samples, n_features))
    w_true = rng.normal(size=n_features)
    y = X @ w_true + rng.normal(scale=noise_std, size=n_samples)

    # Стандартизация признаков для более стабильной оптимизации
    X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-12)
    return X, y, w_true


def mse_loss(X, y, w):
    err = X @ w - y
    return np.mean(err ** 2)


def full_gradient(X, y, w):
    n = X.shape[0]
    err = X @ w - y
    return (2.0 / n) * (X.T @ err)


def batch_gradient(X_batch, y_batch, w):
    n = X_batch.shape[0]
    err = X_batch @ w - y_batch
    return (2.0 / n) * (X_batch.T @ err)


def closed_form_solution(X, y):
    # Псевдообратная матрица для устойчивости на высоких размерностях
    return np.linalg.pinv(X) @ y

## 2) Реализация оптимизаторов (GD, SGD, Momentum, RMSprop, Adam)

In [ ]:
def train_gd(X, y, lr=0.05, epochs=300, w0=None):
    n_features = X.shape[1]
    w = np.zeros(n_features) if w0 is None else w0.copy()
    losses = []

    for _ in range(epochs):
        g = full_gradient(X, y, w)
        w -= lr * g
        losses.append(mse_loss(X, y, w))

    return w, np.array(losses)


def train_sgd(X, y, lr=0.01, epochs=300, batch_size=32, w0=None, seed=0):
    rng = np.random.default_rng(seed)
    n_samples, n_features = X.shape
    w = np.zeros(n_features) if w0 is None else w0.copy()
    losses = []

    for _ in range(epochs):
        idx = rng.permutation(n_samples)
        Xs, ys = X[idx], y[idx]

        for start in range(0, n_samples, batch_size):
            end = start + batch_size
            xb, yb = Xs[start:end], ys[start:end]
            g = batch_gradient(xb, yb, w)
            w -= lr * g

        losses.append(mse_loss(X, y, w))

    return w, np.array(losses)


def train_momentum(X, y, lr=0.01, beta=0.9, epochs=300, batch_size=32, w0=None, seed=0):
    rng = np.random.default_rng(seed)
    n_samples, n_features = X.shape
    w = np.zeros(n_features) if w0 is None else w0.copy()
    v = np.zeros_like(w)
    losses = []

    for _ in range(epochs):
        idx = rng.permutation(n_samples)
        Xs, ys = X[idx], y[idx]

        for start in range(0, n_samples, batch_size):
            end = start + batch_size
            xb, yb = Xs[start:end], ys[start:end]
            g = batch_gradient(xb, yb, w)
            v = beta * v + (1 - beta) * g
            w -= lr * v

        losses.append(mse_loss(X, y, w))

    return w, np.array(losses)


def train_rmsprop(X, y, lr=0.01, beta=0.9, eps=1e-8, epochs=300, batch_size=32, w0=None, seed=0):
    rng = np.random.default_rng(seed)
    n_samples, n_features = X.shape
    w = np.zeros(n_features) if w0 is None else w0.copy()
    s = np.zeros_like(w)
    losses = []

    for _ in range(epochs):
        idx = rng.permutation(n_samples)
        Xs, ys = X[idx], y[idx]

        for start in range(0, n_samples, batch_size):
            end = start + batch_size
            xb, yb = Xs[start:end], ys[start:end]
            g = batch_gradient(xb, yb, w)
            s = beta * s + (1 - beta) * (g ** 2)
            w -= lr * g / (np.sqrt(s) + eps)

        losses.append(mse_loss(X, y, w))

    return w, np.array(losses)


def train_adam(
    X,
    y,
    lr=0.01,
    beta1=0.9,
    beta2=0.999,
    eps=1e-8,
    epochs=300,
    batch_size=32,
    w0=None,
    seed=0,
):
    rng = np.random.default_rng(seed)
    n_samples, n_features = X.shape
    w = np.zeros(n_features) if w0 is None else w0.copy()
    m = np.zeros_like(w)
    v = np.zeros_like(w)
    t = 0
    losses = []

    for _ in range(epochs):
        idx = rng.permutation(n_samples)
        Xs, ys = X[idx], y[idx]

        for start in range(0, n_samples, batch_size):
            end = start + batch_size
            xb, yb = Xs[start:end], ys[start:end]
            g = batch_gradient(xb, yb, w)

            t += 1
            m = beta1 * m + (1 - beta1) * g
            v = beta2 * v + (1 - beta2) * (g ** 2)

            m_hat = m / (1 - beta1 ** t)
            v_hat = v / (1 - beta2 ** t)

            w -= lr * m_hat / (np.sqrt(v_hat) + eps)

        losses.append(mse_loss(X, y, w))

    return w, np.array(losses)

## 3) Запуск эксперимента на одной задаче

In [ ]:
X, y, w_true = make_regression_data(n_samples=2000, n_features=100, noise_std=0.2, seed=1)
w_opt = closed_form_solution(X, y)
opt_loss = mse_loss(X, y, w_opt)

results_single = {}

results_single['GD'] = train_gd(X, y, lr=0.08, epochs=250)
results_single['SGD'] = train_sgd(X, y, lr=0.02, epochs=250, batch_size=64, seed=1)
results_single['Momentum'] = train_momentum(X, y, lr=0.03, beta=0.9, epochs=250, batch_size=64, seed=1)
results_single['RMSprop'] = train_rmsprop(X, y, lr=0.01, beta=0.9, epochs=250, batch_size=64, seed=1)
results_single['Adam'] = train_adam(X, y, lr=0.02, epochs=250, batch_size=64, seed=1)

summary_rows = []
for name, (w_hat, losses) in results_single.items():
    final_loss = losses[-1]
    gap = final_loss - opt_loss
    w_error = np.linalg.norm(w_hat - w_opt)

    threshold = opt_loss + 1e-2
    hit_epochs = np.where(losses <= threshold)[0]
    epochs_to_threshold = int(hit_epochs[0] + 1) if len(hit_epochs) else np.nan

    summary_rows.append({
        'method': name,
        'final_loss': final_loss,
        'loss_gap_to_opt': gap,
        '||w - w*||_2': w_error,
        'epochs_to_(opt+1e-2)': epochs_to_threshold,
    })

single_df = pd.DataFrame(summary_rows).sort_values('final_loss')
single_df

In [ ]:
plt.figure(figsize=(9, 5))
for name, (_, losses) in results_single.items():
    plt.plot(losses, label=name)
plt.axhline(opt_loss, color='black', linestyle='--', label='optimal loss (closed-form)')
plt.yscale('log')
plt.xlabel('Epoch')
plt.ylabel('MSE (log scale)')
plt.title('Сходимость оптимизаторов на одной задаче (d=100)')
plt.legend()
plt.show()

## 4) Эксперименты для разных размерностей

Для каждой размерности:
- генерируется синтетическая задача;
- запускаются Momentum, RMSprop, Adam;
- считаются метрики: итоговый loss, разница до оптимального loss, ошибка по параметрам и эпохи до порога.

In [ ]:
def run_dimension_experiment(dim, seed=0, epochs=300):
    n_samples = max(3000, 20 * dim)
    X, y, _ = make_regression_data(n_samples=n_samples, n_features=dim, noise_std=0.2, seed=seed)
    w_opt = closed_form_solution(X, y)
    opt_loss = mse_loss(X, y, w_opt)

    methods = {
        'Momentum': lambda: train_momentum(X, y, lr=0.03, beta=0.9, epochs=epochs, batch_size=128, seed=seed),
        'RMSprop': lambda: train_rmsprop(X, y, lr=0.01, beta=0.9, epochs=epochs, batch_size=128, seed=seed),
        'Adam': lambda: train_adam(X, y, lr=0.02, epochs=epochs, batch_size=128, seed=seed),
    }

    rows = []
    losses_map = {}

    for name, trainer in methods.items():
        w_hat, losses = trainer()
        losses_map[name] = losses

        final_loss = losses[-1]
        gap = final_loss - opt_loss
        w_error = np.linalg.norm(w_hat - w_opt)

        threshold = opt_loss + 1e-2
        hit_epochs = np.where(losses <= threshold)[0]
        epochs_to_threshold = int(hit_epochs[0] + 1) if len(hit_epochs) else np.nan

        rows.append({
            'dim': dim,
            'method': name,
            'opt_loss': opt_loss,
            'final_loss': final_loss,
            'loss_gap_to_opt': gap,
            '||w - w*||_2': w_error,
            'epochs_to_(opt+1e-2)': epochs_to_threshold,
        })

    return pd.DataFrame(rows), losses_map, opt_loss


dimensions = [20, 100, 500]
all_tables = []
all_curves = {}

for i, d in enumerate(dimensions):
    table_d, curves_d, opt_d = run_dimension_experiment(d, seed=10 + i, epochs=300)
    all_tables.append(table_d)
    all_curves[d] = (curves_d, opt_d)

multi_df = pd.concat(all_tables, ignore_index=True)
multi_df.sort_values(['dim', 'final_loss'])

In [ ]:
fig, axes = plt.subplots(1, len(dimensions), figsize=(16, 4), sharey=True)

for ax, d in zip(axes, dimensions):
    curves_d, opt_d = all_curves[d]
    for name, losses in curves_d.items():
        ax.plot(losses, label=name)
    ax.axhline(opt_d, color='black', linestyle='--', linewidth=1)
    ax.set_title(f'd = {d}')
    ax.set_xlabel('Epoch')
    ax.set_yscale('log')

axes[0].set_ylabel('MSE (log scale)')
axes[-1].legend(loc='upper right')
fig.suptitle('Сходимость Momentum / RMSprop / Adam для разных размерностей')
plt.tight_layout()
plt.show()

## 5) Краткие эмпирические выводы

По типичному запуску на синтетических данных наблюдается:

1. **Adam** часто быстрее остальных достигает малого loss в ранние эпохи за счёт адаптивного шага + момента первого порядка.
2. **RMSprop** обычно стабилен и близок к Adam, но может сходиться чуть медленнее и чувствительнее к `lr`.
3. **Momentum** хорошо ускоряет SGD относительно базового стохастического спуска, но на высоких размерностях чаще требует более аккуратной настройки `lr`.
4. С ростом размерности (`d`) различия в скорости сходимости между адаптивными методами и простым моментом обычно становятся заметнее.
5. По точности финального решения (при достаточном числе эпох) все методы могут приближаться к оптимуму, но число эпох и чувствительность к гиперпараметрам различаются.

> Для воспроизводимого отчёта можно зафиксировать несколько seed и усреднить метрики по прогонам.